# Cognition – Anatomical (CT) PLSc LC Brain Plotting

In [ ]:
!apt-get update
!apt-get install -qq xvfb libgl1-mesa-glx
!pip install pyvista nilearn nibabel -qq
!pip install git+https://github.com/liuzhenqi77/ENIGMA.git@fix-dep

In [ ]:
import pyvista as pv
# Seems that only static plotting is supported by colab at the moment
pv.global_theme.jupyter_backend = 'static'
pv.global_theme.notebook = True
pv.start_xvfb()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as sstats
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
import h5py

In [ ]:
# Set paths (edit to match your local layout)
data_dir = Path("./outputs/anat_cog")  # <-- CHANGE THIS
figs_dir = Path("./figures")
csv_dir = Path("./data")
anat_ep_dir = Path("./outputs/anat")

## Load Data

In [ ]:
# Main PLS result (for scores and cognitive domain loadings)
cog_pls_result = h5py.File(data_dir / "Cog_Anat_pls_result.hdf5")

cog_domains = [
    'General Cognitive Ability',
    'Working Memory',
    'Processing Speed',
    'Sensorimotor Functioning',
    'Language Functioning',
    'Visualspatial Functioning',
    'Executive Functioning'
]

In [ ]:
import matplotlib
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
%matplotlib inline
plt.rcParams['font.size'] = 7
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "path"

from matplotlib.colors import ListedColormap, to_rgb, to_hex
pal_bgo_3 = ["#31A9E7", "#305861", "#F19100"]

In [ ]:
cog_varexp = cog_pls_result["results/varexp"][:]
cog_varexp_pval = cog_pls_result["results/permres/pvals"][:]
cog_varexp_pval_color = ["tab:red" if p <= 0.05 else "silver" for p in cog_varexp_pval]

print("LC | Var. Explained | P-value")
for i in range(len(cog_varexp)):
    print(f" {i+1} |    {cog_varexp[i]:.4f}     | {cog_varexp_pval[i]:.4f} {'*' if cog_varexp_pval[i] <= 0.05 else ''}")

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(range(len(cog_varexp)), cog_varexp, s=15, c=cog_varexp_pval_color)


ax.set(xticks=[0, 1, 2, 3, 4, 5, 6], yticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
ax.set_xlabel("Latent components", fontsize=10)
ax.set_ylabel("Covariance explained", fontsize=10)

sns.despine(top=True, right=True, trim=False, ax=ax)
ax.tick_params(axis=u'both', which=u'both', length=1)
fig.savefig(figs_dir / "cog_anat_exp_var.svg")

In [ ]:
from scipy import stats

x_scores = cog_pls_result["results"]["x_scores"][:, 0]
y_scores = cog_pls_result["results"]["y_scores"][:, 0]

fig, ax = plt.subplots(figsize=(3, 3), layout="constrained")
scatter = ax.scatter(x_scores, y_scores,
                     s=18, color=pal_bgo_3[1], linewidths=0.1, edgecolors='white')
ax.axvline(x=0, c="silver", zorder=0, ls=(0, (2, 1)))
ax.axhline(y=0, c="silver", zorder=0, ls=(0, (2, 1)))

slope, intercept, r_value, p_value, std_err = stats.linregress(x_scores, y_scores)
x_line = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1], 100)
y_line = slope * x_line + intercept
ax.plot(x_line, y_line, color=pal_bgo_3[1], linestyle='-', linewidth=1.5, zorder=1)


ax.set(xlim=(-17, 12), ylim=(-5, 8), xticks=[-15, -10, -5, 0, 5, 10, 15], yticks=[-4, -2, 0, 2, 4, 6, 8])

ax.set_xlabel("Anatomical composite scores", fontsize=10)
ax.set_ylabel("Cognitive composite scores", fontsize=10)

score_corr_r, score_corr_p = sstats.pearsonr(x_scores, y_scores)
ax.text(-13, 4, f"r={score_corr_r:.2f}\np={score_corr_p:.2e}", fontsize=10)

sns.despine(top=True, right=True, trim=True, ax=ax)
ax.tick_params(axis=u'both', which=u'both', length=1)

fig.savefig(figs_dir / "cog_anat_score_corr.svg")

In [ ]:
cog_pls_yload = cog_pls_result["results/y_loadings"][:, 0]
cog_pls_yload_err = (
    cog_pls_result["results/bootres/y_loadings_ci"][:, 0, 1]
    - cog_pls_result["results/bootres/y_loadings_ci"][:, 0, 0]
) / 2

cog_pls_yload_sig_mask = (np.abs(cog_pls_yload) - cog_pls_yload_err) > 0

print("Cognitive Domain Loadings (LC1):")
for i in range(len(cog_domains)):
    sig = "*" if cog_pls_yload_sig_mask[i] else ""
    print(f"  {cog_domains[i]:<35} {cog_pls_yload[i]:>8.4f} \u00b1 {cog_pls_yload_err[i]:.4f} {sig}")

In [ ]:
temp = pd.DataFrame({
    'Cognitive Domain': cog_domains,
    'Loading': cog_pls_yload,
    'SE': cog_pls_yload_err,
    'Loading (SE)': [f"{cog_pls_yload[i]:.3f} ({cog_pls_yload_err[i]:.3f})" for i in range(7)],
    'Significant': cog_pls_yload_sig_mask
})
display(temp[['Cognitive Domain', 'Loading (SE)', 'Significant']])

In [ ]:
cog_color = "#2E86AB"

all_resort = np.argsort(-1 * cog_pls_yload)

plot_x = np.arange(len(cog_domains))
plot_y = cog_pls_yload[all_resort]
plot_error = cog_pls_yload_err[all_resort]
plot_labels = np.array(cog_domains)[all_resort]
plot_sig = cog_pls_yload_sig_mask[all_resort]
plot_colors = [cog_color if s else "lightgray" for s in plot_sig]

plot_y_pos = np.where(plot_y > 0)[0]
plot_y_neg = np.where(plot_y <= 0)[0]

fig, ax = plt.subplots(figsize=(3.5, 2.5), layout="constrained")

ax.axvline(x=0, c="black", lw=0.5)

ax.barh(plot_x, plot_y, xerr=plot_error, color=plot_colors, height=0.75,
        error_kw=dict(ecolor="lightgray", lw=1, capsize=2, capthick=1))

ax.set(yticks=plot_x, yticklabels=[])
ax.invert_yaxis()
sns.despine(top=True, right=True, left=True, ax=ax)
ax.tick_params(axis=u'both', which=u'both', length=0)

for i in plot_y_pos:
    ax.text(-0.02, plot_x[i], plot_labels[i], ha="right", va="center", color="k")
for i in plot_y_neg:
    ax.text(0.02, plot_x[i], plot_labels[i], ha="left", va="center", color="k")

ax.set_xlabel("Cognitive Loading", fontsize=10)
fig.savefig(figs_dir / "cog_anat_yload.svg")

In [ ]:
cog_color = "#2E86AB"

cog_pls_yload_all_idx = np.where(cog_pls_yload_sig_mask)[0]
cog_pls_yload_all_resort = np.argsort(-1 * cog_pls_yload[cog_pls_yload_all_idx])

plot_x = np.arange(len(cog_pls_yload_all_resort))
plot_y = cog_pls_yload[cog_pls_yload_all_idx][cog_pls_yload_all_resort]
plot_error = cog_pls_yload_err[cog_pls_yload_all_idx][cog_pls_yload_all_resort]
plot_labels = np.array(cog_domains)[cog_pls_yload_all_idx][cog_pls_yload_all_resort]

plot_y_pos = np.where(plot_y > 0)[0]
plot_y_neg = np.where(plot_y <= 0)[0]

fig, ax = plt.subplots(figsize=(3.5, 0.75), layout="constrained")

ax.axvline(x=0, c="black", lw=0.5)

ax.barh(plot_x, plot_y, xerr=plot_error, color=cog_color, height=0.75,
        error_kw=dict(ecolor="lightgray", lw=1, capsize=2, capthick=1))

ax.set(yticks=plot_x, yticklabels=[])
ax.invert_yaxis()
sns.despine(top=True, right=True, left=True, ax=ax)
ax.tick_params(axis=u'both', which=u'both', length=0)

for i in plot_y_pos:
    ax.text(-0.02, plot_x[i], plot_labels[i], ha="right", va="center", color="k")
for i in plot_y_neg:
    ax.text(0.02, plot_x[i], plot_labels[i], ha="left", va="center", color="k")

ax.set_xlabel("Cognitive Loading", fontsize=10)
fig.savefig(figs_dir / "cog_anat_yload.svg")

In [ ]:
# Region names (same 84 features: 16 subcortical + 68 cortical DK)
pls_anat_input = pd.read_csv(csv_dir / "PLS_all_cog_anat_CT.csv")
pls_anat_input_names = pls_anat_input.columns[-84:]


In [ ]:
# Load PLS loadings — one file for cognition (all 7 domains as Y)
# From flipped PLS: pyls.behavioral_pls(Y, X)
# y_loadings here = brain loadings (84 regions)
cog_pls_loadings = h5py.File(data_dir / "Cog_Anat_pls_loadings.hdf5")

In [ ]:
from enigmatoolbox.datasets import load_summary_stats
from enigmatoolbox.utils.parcellation import parcel_to_surface, subcorticalvertices

## Compute Significance & Extract Cortical/Subcortical Loadings

In [ ]:
# Bootstrap CI for significance
cog_yload_ci0 = cog_pls_loadings['results/bootres/y_loadings_ci'][:, 0, 0]
cog_yload_ci1 = cog_pls_loadings['results/bootres/y_loadings_ci'][:, 0, 1]

cog_yload_symerror = (cog_yload_ci1 - cog_yload_ci0) / 2
cog_yload_ci = np.abs(cog_pls_loadings['results/y_loadings'][:, 0]) - cog_yload_symerror
cog_yload_sig = (cog_yload_ci > 0).astype(float)

# Set non-significant to NaN for masked plotting
cog_yload_sig[cog_yload_sig == 0] = np.nan
cog_loadings_sig = cog_pls_loadings['results/y_loadings'][:, 0] * cog_yload_sig

# Split into cortical (68 DK regions) and subcortical (16 Tian)
cog_ctx = cog_loadings_sig[16:]   # 68 cortical
cog_subc = cog_loadings_sig[:16]  # 16 subcortical

print(f"Cortical: {np.sum(~np.isnan(cog_ctx)):.0f}/68 significant regions")
print(f"Subcortical: {np.sum(~np.isnan(cog_subc)):.0f}/16 significant regions")

In [ ]:
# Print regional loadings
print("\nSignificant cortical regions:")
for i in range(68):
    if not np.isnan(cog_ctx[i]):
        print(f"  {pls_anat_input_names[16+i]:<40} {cog_ctx[i]:.4f}")

print("\nSignificant subcortical regions:")
for i in range(16):
    if not np.isnan(cog_subc[i]):
        print(f"  {pls_anat_input_names[i]:<40} {cog_subc[i]:.4f}")

## Map to Surface

In [ ]:
# Reorder subcortical for enigmatoolbox
subc_reorder_index = [14, 12, 4, 10, 8, 6, 2, 0, 15, 13, 5, 11, 9, 7, 3, 1]
print("Subcortical order:", list(pls_anat_input_names[:16][subc_reorder_index]))

In [ ]:
cog_ctx_fsa5 = parcel_to_surface(cog_ctx, 'aparc_fsa5')
cog_subc_fsa5 = subcorticalvertices(cog_subc[subc_reorder_index])

## Surface Plotting

In [ ]:
def make_pv_surf(surf_hemi):
    vertices, faces = nib.load(surf_hemi).agg_data()
    surf = pv.PolyData(vertices, np.c_[np.ones((faces.shape[0],), dtype=int)*3, faces])
    return surf

def make_pv_plotter(surf_lh, surf_rh, plotter_settings, mesh_settings):

    pl = pv.Plotter(shape=(2, 2), border=False, **plotter_settings)

    zoom_ratio = 1.5

    pl.subplot(0, 0)
    pl.add_mesh(surf_lh.rotate_z(180), show_scalar_bar=False, **mesh_settings)
    pl.camera_position="yz"
    pl.zoom_camera(zoom_ratio)

    pl.subplot(1, 0)
    pl.add_mesh(surf_lh, show_scalar_bar=False, **mesh_settings)
    pl.camera_position="yz"
    pl.zoom_camera(zoom_ratio)

    pl.subplot(0, 1)
    pl.add_mesh(surf_rh, show_scalar_bar=False, **mesh_settings)
    pl.camera_position="yz"
    pl.zoom_camera(zoom_ratio)

    pl.subplot(1, 1)
    pl.add_mesh(surf_rh.rotate_z(180), show_scalar_bar=False, **mesh_settings)
    pl.camera_position="yz"
    pl.zoom_camera(zoom_ratio)

    cbar_settings = dict(
        n_labels=2,
        label_font_size=10,
        title_font_size=15,
        font_family="arial",
        width=0.3,
        position_x=0.65,
        height=0.15,
    )

    cbar = pl.add_scalar_bar(**cbar_settings)

    light = pv.Light(light_type='headlight', intensity=0.2)
    pl.add_light(light)

    return pl

In [ ]:
# Load surfaces — using the plotting files from symptom anat dir
surf_lh = make_pv_surf(anat_ep_dir / "plotting/fsa5_lh.gii")
surf_rh = make_pv_surf(anat_ep_dir / "plotting/fsa5_rh.gii")

# Cognition cortical loadings
surf_lh.point_data['cog_ctx_fsa5'] = cog_ctx_fsa5[:10242]
surf_rh.point_data['cog_ctx_fsa5'] = cog_ctx_fsa5[10242:]

# Non-significant mask (for showing which regions are non-significant)
cog_ctx_fsa5_zeros = np.zeros_like(cog_ctx_fsa5)
cog_ctx_fsa5_zeros[np.where(cog_ctx_fsa5 == 0)] = 1
surf_lh.point_data['cog_ctx_fsa5_zeros'] = cog_ctx_fsa5_zeros[:10242]
surf_rh.point_data['cog_ctx_fsa5_zeros'] = cog_ctx_fsa5_zeros[10242:]

# Subcortical surfaces
subc_lh = make_pv_surf(anat_ep_dir / "plotting/sctx_lh.gii")
subc_rh = make_pv_surf(anat_ep_dir / "plotting/sctx_rh.gii")

subc_lh.point_data['cog_subc_fsa5'] = cog_subc_fsa5[:25910]
subc_rh.point_data['cog_subc_fsa5'] = cog_subc_fsa5[25910:]

plotter_settings = dict(
    notebook=True,
    window_size=(400, 300)
)

### Cortical — Non-significant Regions Mask

In [ ]:
mesh_settings = dict(
    scalars="cog_ctx_fsa5_zeros",
    smooth_shading=True,
    cmap="Reds",
    clim=(0, 1)
)

pl = make_pv_plotter(surf_lh, surf_rh, plotter_settings, mesh_settings)
pl.show()

### Cortical — Significant Loadings

In [ ]:
mesh_settings = dict(
    scalars="cog_ctx_fsa5",
    smooth_shading=True,
    cmap="RdBu_r",
    clim=(-0.35, 0.35)  # adjust based on data range
)

pl = make_pv_plotter(surf_lh, surf_rh, plotter_settings, mesh_settings)
pl.show()

### Subcortical — Significant Loadings

In [ ]:
mesh_settings = dict(
    scalars="cog_subc_fsa5",
    smooth_shading=True,
    split_sharp_edges=True,
    cmap="RdBu_r",
    clim=(-0.3, 0.3)  # adjust based on data range
)

pl = make_pv_plotter(subc_lh, subc_rh, plotter_settings, mesh_settings)
pl.show()

In [ ]:
# Create tables of brain region names and their significant PLS loadings

# Positive symptoms
cog_table = pd.DataFrame({
    "Region": pls_anat_input_names,
    "PLS Loading": cog_loadings_sig,
    "Significant": ~np.isnan(cog_loadings_sig)
})
cog_table["Type"] = ["Subcortical"]*16 + ["Cortical"]*68
cog_table = cog_table.sort_values("PLS Loading", key=abs, ascending=False, na_position="last")

print("=== Cognitive Symptoms PLS Loadings ===")
print(f"Significant regions: {cog_table['Significant'].sum():.0f} / {len(cog_table)}")
display(cog_table.reset_index(drop=True))

In [ ]:
cog_table.to_csv(csv_dir / "PLS_cog_anat_loadings_table.csv", index=False)